In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
output_path = 'MathV3(markov).csv'

# =============================================================================
# Core Function: Spatiotemporal Markov Model (Ultimate Dual-Layer)
# =============================================================================
def process_single_user_unified(user_df):
    uid = user_df['uid'].iloc[0]
    
    # 1. 准备训练数据与全网格填充
    train_df = user_df[user_df['d'] <= 59].drop_duplicates(subset=['d', 't'], keep='last').copy()
    if train_df.empty: return pd.DataFrame()

    days, ts = range(0, 60), range(48)
    full_index = pd.MultiIndex.from_product([[uid], days, ts], names=['uid', 'd', 't'])
    train_full = train_df.set_index(['uid', 'd', 't']).reindex(full_index)
    train_full = train_full.ffill().bfill().reset_index()
    train_full['weekday'] = train_full['d'] % 7

    # 2. 建立转移状态
    train_full['prev_x'] = train_full['x'].shift(1)
    train_full['prev_y'] = train_full['y'].shift(1)
    transitions = train_full.dropna(subset=['prev_x', 'prev_y']).copy()

    # --- 第一层防线：完整马尔可夫转移规律 ---
    matrix_full = transitions.groupby(['weekday', 't', 'prev_x', 'prev_y', 'x', 'y']).size().reset_index(name='count')
    best_full = matrix_full.sort_values('count', ascending=False).drop_duplicates(subset=['weekday', 't', 'prev_x', 'prev_y'])

    # --- 第二层防线：纯时间作息规律（因为数据已填充，此层必有解） ---
    matrix_temp = train_full.groupby(['weekday', 't', 'x', 'y']).size().reset_index(name='count')
    best_temp = matrix_temp.sort_values('count', ascending=False).drop_duplicates(subset=['weekday', 't'])

    # 3. 递归预测 (Days 60-74)
    curr_x, curr_y = train_full.iloc[-1]['x'], train_full.iloc[-1]['y']
    results = []

    for d in range(60, 75):
        weekday = d % 7
        for t in range(48):
            # 尝试第一层：顺着物理轨迹走
            match = best_full[(best_full['weekday'] == weekday) & (best_full['t'] == t) & 
                              (best_full['prev_x'] == curr_x) & (best_full['prev_y'] == curr_y)]
            
            if not match.empty:
                next_x, next_y = match.iloc[0]['x'], match.iloc[0]['y']
            else:
                # 尝试第二层：轨迹断了，直接靠作息生物钟归位 (必然命中)
                match_t = best_temp[(best_temp['weekday'] == weekday) & (best_temp['t'] == t)]
                next_x, next_y = match_t.iloc[0]['x'], match_t.iloc[0]['y']
            
            results.append([uid, d, t, int(next_x), int(next_y)])
            curr_x, curr_y = next_x, next_y # 更新当前位置，继续往下推导

    return pd.DataFrame(results, columns=['uid', 'd', 't', 'x', 'y'])

# =============================================================================
# Execution Engine (Streaming)
# =============================================================================
pd.DataFrame(columns=['uid', 'd', 't', 'x', 'y']).to_csv(output_path, index=False)
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}
buffer_df = pd.DataFrame()

TARGET_USERS = 5000
user_count, stop_reading = 0, False

print(f"Starting Ultimate Dual-Layer Markov Model for {TARGET_USERS} users...")

for chunk in pd.read_csv(path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=500000):
    if stop_reading: break
    chunk = pd.concat([buffer_df, chunk], ignore_index=True)
    unique_uids = chunk['uid'].unique()
    if len(unique_uids) == 1: buffer_df = chunk; continue
    
    last_uid = unique_uids[-1]
    complete_data, buffer_df = chunk[chunk['uid'] != last_uid], chunk[chunk['uid'] == last_uid]
    
    for uid, user_data in complete_data.groupby('uid'):
        if user_count >= TARGET_USERS: stop_reading = True; break
        user_pred = process_single_user_unified(user_data)
        if not user_pred.empty:
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
        user_count += 1
        if user_count % 500 == 0: print(f"Progress: {user_count} / {TARGET_USERS}...")

if not stop_reading and not buffer_df.empty and user_count < TARGET_USERS:
    for uid, user_data in buffer_df.groupby('uid'):
        user_pred = process_single_user_unified(user_data)
        if not user_pred.empty: user_pred.to_csv(output_path, mode='a', header=False, index=False)
        user_count += 1

print(f"🎉 Ultimate Markov Predictions saved to: {output_path}")

Starting Ultimate Dual-Layer Markov Model for 5000 users...
Progress: 500 / 5000...
Progress: 1000 / 5000...
Progress: 1500 / 5000...
Progress: 2000 / 5000...
Progress: 2500 / 5000...
Progress: 3000 / 5000...
Progress: 3500 / 5000...
Progress: 4000 / 5000...
Progress: 4500 / 5000...
Progress: 5000 / 5000...
🎉 Ultimate Markov Predictions saved to: MathV3(markov).csv


In [2]:
pred_path = 'MathV3(markov).csv'
orig_path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'

print("1. Loading predictions into memory...")
# Load the predictions we just generated
pred_df = pd.read_csv(pred_path)
target_uids = pred_df['uid'].unique()
print(f"-> Successfully loaded predictions for {len(target_uids)} users.")

print("\n2. Extracting Ground Truth (Days 60-74) from massive dataset...")
gt_chunks = []
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}

# Stream original data to prevent Memory Crash
for chunk in pd.read_csv(orig_path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=1000000):
    
    # Filter 1: Only keep Days 60 to 74
    chunk = chunk[(chunk['d'] >= 60) & (chunk['d'] <= 74)]
    
    # Filter 2: Only keep users that we actually predicted
    chunk = chunk[chunk['uid'].isin(target_uids)]
    
    # Remove duplicates just in case
    chunk = chunk.drop_duplicates(subset=['uid', 'd', 't'], keep='last')
    
    if not chunk.empty:
        gt_chunks.append(chunk)

# Combine all valid chunks into the final Ground Truth dataframe
gt_df = pd.concat(gt_chunks, ignore_index=True)
print(f"-> Extracted {len(gt_df)} real trajectory records for evaluation.")

print("\n3. Comparing Predictions vs. Ground Truth...")
# Merge Ground Truth and Predictions on UID, Day, and Time Bin
# suffix _true for actual data, _pred for our model's data
eval_df = pd.merge(gt_df, pred_df, on=['uid', 'd', 't'], suffixes=('_true', '_pred'))

# A prediction is correct ONLY if both X and Y coordinates match exactly
eval_df['is_correct'] = (eval_df['x_true'] == eval_df['x_pred']) & (eval_df['y_true'] == eval_df['y_pred'])

# Calculate Accuracy (Correct predictions / Total evaluated records)
accuracy = eval_df['is_correct'].mean() * 100
total_evaluated = len(eval_df)
correct_count = eval_df['is_correct'].sum()

print("==================================================")
print("📊 EVALUATION RESULTS")
print("==================================================")
print(f"Total Evaluated Records: {total_evaluated}")
print(f"Correct Predictions:     {correct_count}")
print(f"Overall Accuracy:        {accuracy:.2f} %")
print("==================================================")

# Optional: View where the model failed
# errors = eval_df[eval_df['is_correct'] == False]
# print(errors.head())

1. Loading predictions into memory...
-> Successfully loaded predictions for 5000 users.

2. Extracting Ground Truth (Days 60-74) from massive dataset...
-> Extracted 1251954 real trajectory records for evaluation.

3. Comparing Predictions vs. Ground Truth...
📊 EVALUATION RESULTS
Total Evaluated Records: 1251954
Correct Predictions:     247076
Overall Accuracy:        19.74 %
